# Projeto Classificador de Imagens: Comites de Classificadores

## Imports e funções

In [1]:
import time 
import joblib
import multiprocessing
joblib.parallel_backend('threading')
print(f"Número de CPUs disponíveis: {multiprocessing.cpu_count()}")
import pandas   as pd
import numpy    as np

from numpy                      import mean
from numpy                      import std
from sklearn                    import metrics
from sklearn.metrics            import confusion_matrix, f1_score
from sklearn.preprocessing      import minmax_scale

from sklearn.model_selection    import train_test_split, KFold, cross_val_score, cross_val_predict, ParameterGrid

from sklearn.naive_bayes        import GaussianNB, MultinomialNB, ComplementNB
from sklearn.neural_network     import MLPClassifier
from sklearn.neighbors          import KNeighborsClassifier
from sklearn.tree               import DecisionTreeClassifier

from sklearn.linear_model       import LogisticRegression

from sklearn.ensemble           import BaggingClassifier
from sklearn.ensemble           import RandomForestClassifier
from sklearn.ensemble           import VotingClassifier
from sklearn.ensemble           import StackingClassifier

import traceback
import warnings
import ast
from sklearn.exceptions         import ConvergenceWarning
from datetime                   import datetime

from IPython.display import display

# Parametros gerais para homogeneidade de metodologia
# Tamanho do conjunto de teste
tamanho_ds_teste=0.2
semente_aleatoria=42
versao_base = "v20251102-1210"
paralelismo = -1

def raca_para_especie(raca):
    if raca in ['basset_hound', 'saint_bernard']:
        return 'dog'
    elif raca in ['Birman', 'Persian']:
        return 'cat'
    else:
        return raca  # fallback

def separar_dataset(df, scale=False):
    X = df.iloc[:, :-1]
    ### Para Bases com PCA que geram valores negativos permite normalizar os valores
    if scale:
        X = minmax_scale(X)
    y = df.iloc[:, -1]
    return X, y

def get_scale_setting(params, dataset_key):
    """
    Determina se deve escalar baseado no modelo e parâmetros
    """
    nb_type = params.get('nb_type')
    if nb_type in ['MultinomialNB', 'ComplementNB']:
        if dataset_key and dataset_key.endswith('_pca'):
            return True
    return False
            


Número de CPUs disponíveis: 12


## Resultados dos testes de classificadores base

In [2]:
# KNN DataFrame
knn_configs = pd.DataFrame({
    'params': [
        {'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'distance'},
        {'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'uniform'},
        {'metric': 'chebyshev', 'n_neighbors': 5, 'weights': 'distance'},
        {'metric': 'chebyshev', 'n_neighbors': 5, 'weights': 'uniform'},
        {'metric': 'chebyshev', 'n_neighbors': 11, 'weights': 'distance'},
        {'metric': 'chebyshev', 'n_neighbors': 11, 'weights': 'uniform'},
        {'metric': 'chebyshev', 'n_neighbors': 15, 'weights': 'distance'},
        {'metric': 'chebyshev', 'n_neighbors': 15, 'weights': 'uniform'},
        {'metric': 'chebyshev', 'n_neighbors': 7, 'weights': 'distance'},
        {'metric': 'chebyshev', 'n_neighbors': 7, 'weights': 'uniform'}
    ],
    'f1_score': [0.767689, 0.767689, 0.762123, 0.762123, 0.759038, 0.759038, 0.751327, 0.751327, 0.747914, 0.747914]
})

# Decision Tree DataFrame
dtree_configs = pd.DataFrame({
    'params': [
        {'criterion': 'entropy', 'max_depth': 3},
        {'criterion': 'log_loss', 'max_depth': 3},
        {'criterion': 'gini', 'max_depth': 3},
        {'criterion': 'entropy', 'max_depth': 5},
        {'criterion': 'log_loss', 'max_depth': 4},
        {'criterion': 'entropy', 'max_depth': 4},
        {'criterion': 'gini', 'max_depth': 4},
        {'criterion': 'log_loss', 'max_depth': 5},
        {'criterion': 'entropy', 'max_depth': 10},
        {'criterion': 'log_loss', 'max_depth': 9}
    ],
    'f1_score': [0.756682, 0.756682, 0.729409, 0.725778, 0.724054, 0.724054, 0.723451, 0.719643, 0.691604, 0.675000]
})

# Naive Bayes DataFrame
nb_configs = pd.DataFrame({
    'params': [
        {'nb_type': 'ComplementNB'},
        {'nb_type': 'GaussianNB'},
        {'nb_type': 'MultinomialNB'}
    ],
    'f1_score': [0.744524, 0.713310, 0.657967],
    'config_rank': [1, 0, 2],
    'status': ['sucesso'] * 3
})

# MLP DataFrame
mlp_configs = pd.DataFrame({
    'params': [
        {'activation': 'relu', 'hidden_layer_sizes': (200, 100),         'learning_rate_init': 0.005, 'max_iter': 500,  'solver': 'adam'},
        {'activation': 'logistic', 'hidden_layer_sizes': (200, 100, 50), 'learning_rate_init': 0.005, 'max_iter': 500,  'solver': 'adam'},
        {'activation': 'relu', 'hidden_layer_sizes': 50,                 'learning_rate_init': 0.01,  'max_iter': 500,  'solver': 'adam'},
        {'activation': 'tanh', 'hidden_layer_sizes': (150, 75),          'learning_rate_init': 0.01,  'max_iter': 500,  'solver': 'adam'},
        {'activation': 'relu', 'hidden_layer_sizes': (200, 100),         'learning_rate_init': 0.01,  'max_iter': 500,  'solver': 'adam'},
        {'activation': 'relu', 'hidden_layer_sizes': (200, 100, 50),     'learning_rate_init': 0.01,  'max_iter': 500,  'solver': 'sgd'},
        {'activation': 'identity', 'hidden_layer_sizes': (200, 100, 50), 'learning_rate_init': 0.001, 'max_iter': 500,  'solver': 'adam'},
        {'activation': 'relu', 'hidden_layer_sizes': (200, 100, 50),     'learning_rate_init': 0.005, 'max_iter': 500,  'solver': 'adam'},
        {'activation': 'relu', 'hidden_layer_sizes': 150,                'learning_rate_init': 0.01,  'max_iter': 500,  'solver': 'adam'},
        {'activation': 'relu', 'hidden_layer_sizes': (200, 100),         'learning_rate_init': 0.001, 'max_iter': 1500, 'solver': 'adam'}
    ],
    'f1_score': [0.806836, 0.787968, 0.787163, 0.781638, 0.781397, 0.775636, 0.774643, 0.771721, 0.769341, 0.768905],
    'status': ['sucesso'] * 10
})

# Dicionário principal
top10_configs = {
    'knn': {
        'class': KNeighborsClassifier,
        'all_configs': knn_configs,
        'fixed_params': {},
        'model_name': 'KNN'
    },
    'dtree': {
        'class': DecisionTreeClassifier,
        'all_configs': dtree_configs,
        'fixed_params': {},
        'model_name': 'Decision Tree'
    },
    'nb': {
        'class': [GaussianNB, MultinomialNB, ComplementNB],
        'all_configs': nb_configs,
        'fixed_params': {},
        'model_name': 'Naive Bayes'
    },
    'mlp': {
        'class': MLPClassifier,
        'all_configs': mlp_configs,
        'fixed_params': {
            'random_state': semente_aleatoria,
        },
        'model_name': 'MLP'
    }
}

## Preparando Arquivos


### Lendo lista de datasets

In [3]:
datafiles = pd.read_csv('../dataset_list.csv',encoding='utf-8')

datafiles.head(20)

,key,filename
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz


### Lendo Dataframes e ajustando dados

In [4]:

dfs = {}
shapes = []
last_col = []

for metadata in datafiles.itertuples():
    # Imprime o arquivo que está sendo lido
    #print(f"Lendo arquivo: {metadata.key}")
    # Carrega o DataFrame
    df = pd.read_csv(metadata.filename)
    
    # Aplica a função raca_para_especie na coluna raca
    if 'raca' in df.columns:
        df['especie'] = df['raca'].apply(raca_para_especie)
        df = df.drop('raca', axis=1)
    
    # Elimina a coluna nome_arquivo se existir
    if 'nome_arquivo' in df.columns:
        df = df.drop('nome_arquivo', axis=1)
    
    dfs[metadata.key] = df
    shapes.append(df.shape)
    # Pega a última coluna
    last_col.append(df.columns[-1:].tolist())

# Adiciona as colunas shape e last_col ao datafiles
datafiles['shape'] = shapes
datafiles['last_column'] = last_col

datafiles.head(12)    

,key,filename,shape,last_column
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz,"(800, 104)",[especie]
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz,"(800, 649)",[especie]
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz,"(800, 325)",[especie]
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz,"(800, 3601)",[especie]
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz,"(800, 94)",[especie]
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz,"(800, 118)",[especie]
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz,"(800, 325)",[especie]
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz,"(800, 51)",[especie]
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz,"(800, 99)",[especie]
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz,"(800, 1765)",[especie]


## Parametros de configuração para os comites

In [5]:

TRAINING_TYPES = ["holdout", "crossvalidation"]

df_resultados_recuperados = joblib.load(f'resultados_top10_todos_modelos_{versao_base}.joblib')
# Primeiro, vamos inspecionar a estrutura dos dados carregados
print("Tipo dos dados na coluna 'params':", type(df_resultados_recuperados['params'].iloc[0]))
print("Amostra dos dados na coluna 'params':", df_resultados_recuperados['params'].iloc[0])

# Se params contém dicionários, precisamos convertê-los para strings hasháveis
df_resultados_recuperados['params_str'] = df_resultados_recuperados['params'].astype(str)

df_resultados_recuperados['params_str'] = df_resultados_recuperados['params'].astype(str)

# Criar o dataframe agregado usando a coluna params_str para agrupar
df_agregado = df_resultados_recuperados.groupby(['model', 'model_name', 'config_rank', 'params_str']).agg({
    'f1_score': ['mean', 'std'],
    'execution_time': ['mean', 'std']
}).reset_index()

# Renomear as colunas para os nomes solicitados
df_agregado.columns = ['model', 'model_name', 'config_rank', 'params', 
                       'media_f1_score', 'desvio_padrao_f1_score', 
                       'media_execution_time', 'desvio_padrao_execution_time']

df_agregado = df_agregado.sort_values(by=['model', 'model_name', 'media_f1_score', 'media_execution_time'], 
                                      ascending=[True, True, False, False]).reset_index(drop=True)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None) 
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

df_agregado.head(100)
# df_resultados_recuperados.head(100)

Tipo dos dados na coluna 'params': <class 'dict'>
Amostra dos dados na coluna 'params': {'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'distance'}


,model,model_name,config_rank,params,media_f1_score,desvio_padrao_f1_score,media_execution_time,desvio_padrao_execution_time
0,dtree,Decision Tree,8,"{'criterion': 'log_loss', 'max_depth': 5}",0.676379,0.034280,2.222140,4.770541
1,dtree,Decision Tree,4,"{'criterion': 'entropy', 'max_depth': 5}",0.676242,0.032235,2.252403,4.871926
2,dtree,Decision Tree,7,"{'criterion': 'gini', 'max_depth': 4}",0.676075,0.029364,1.459164,3.111566
3,dtree,Decision Tree,3,"{'criterion': 'gini', 'max_depth': 3}",0.674964,0.038576,1.139408,2.424747
4,dtree,Decision Tree,1,"{'criterion': 'entropy', 'max_depth': 3}",0.674054,0.044310,1.493544,3.184053
5,dtree,Decision Tree,2,"{'criterion': 'log_loss', 'max_depth': 3}",0.673509,0.044417,1.490642,3.186151
6,dtree,Decision Tree,5,"{'criterion': 'log_loss', 'max_depth': 4}",0.672510,0.034564,1.885793,4.038630
7,dtree,Decision Tree,6,"{'criterion': 'entropy', 'max_depth': 4}",0.671537,0.033615,1.896134,4.048257
8,dtree,Decision Tree,10,"{'criterion': 'log_loss', 'max_depth': 9}",0.664883,0.032203,2.847876,6.131907
9,dtree,Decision Tree,9,"{'criterion': 'entropy', 'max_depth': 10}",0.660403,0.030228,2.899146,6.253606


In [6]:
def get_nth_best_model(df, model_identifier, n, search_by='model_name'):
    """
    Retorna o n-ésimo melhor modelo de um tipo específico.
    
    Args:
        df: DataFrame ordenado
        model_identifier: Identificador do modelo (pode ser nome legível ou código)
        n: Posição (1-based, ou seja, 1 = melhor, 2 = segundo melhor, etc.)
        search_by: Coluna para buscar ('model_name' ou 'model')
                  - 'model_name': busca por nomes legíveis (ex: 'MLP', 'KNN')
                  - 'model': busca por códigos (ex: 'mlp', 'knn')
    
    Returns:
        Series com os dados do modelo ou None se a posição não existir
    
    Examples:
        # Buscar 3º melhor MLP por nome legível
        get_nth_best_model(df_agregado, 'MLP', 3, search_by='model_name')
        
        # Buscar 3º melhor MLP por código
        get_nth_best_model(df_agregado, 'mlp', 3, search_by='model')
        
        # Por padrão busca por model_name
        get_nth_best_model(df_agregado, 'MLP', 3)
    """
    if search_by not in ['model', 'model_name']:
        raise ValueError("search_by deve ser 'model' ou 'model_name'")
    
    filtered_df = df[df[search_by] == model_identifier]
    
    # Verificar se existem modelos suficientes
    if len(filtered_df) < n:
        return None
    
    return filtered_df.iloc[n-1]

# Exemplos de uso:

# Caso normal - funciona
terceiro_mlp = get_nth_best_model(df_agregado, 'MLP', 3)
if terceiro_mlp is not None:
    print("3º melhor MLP encontrado:")
    print(terceiro_mlp)
else:
    print("Não existe 3º melhor MLP")

# Caso que retorna None - posição não existe
centesimo_mlp = get_nth_best_model(df_agregado, 'MLP', 100)
if centesimo_mlp is not None:
    print("100º melhor MLP encontrado:")
    print(centesimo_mlp)
else:
    print("Não existe 100º melhor MLP (posição inexistente)")

# Modelo inexistente
modelo_inexistente = get_nth_best_model(df_agregado, 'ModeloQueNaoExiste', 1)
if modelo_inexistente is not None:
    print("Modelo encontrado")
else:
    print("Modelo inexistente")

   

3º melhor MLP encontrado:
model                                                                                                                                               mlp
model_name                                                                                                                                          MLP
config_rank                                                                                                                                           5
params                          {'activation': 'relu', 'hidden_layer_sizes': (200, 100), 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'adam'}
media_f1_score                                                                                                                                  0.75908
desvio_padrao_f1_score                                                                                                                         0.023755
media_execution_time                                          

In [7]:
def get_best_config_by_criterion(df, criterion):
    """
    Retorna a melhor configuração do Decision Tree para um critério específico.
    """
    # Filtrar Decision Trees que usam este critério
    dt_configs = df[(df['model'] == 'dtree') & (df['params'].str.contains(f"'criterion': '{criterion}'"))]
    
    if not dt_configs.empty:
        return dt_configs.iloc[0]  # Melhor configuração (já ordenada)
    else:
        return None

# Usar a função
for criterio in ['gini', 'entropy', 'log_loss']:
    config = get_best_config_by_criterion(df_agregado, criterio)
    if config is not None:
        print(f"{criterio}: {config}")
    else:
        print(f"{criterio}: Não encontrado")

gini: model                                                           dtree
model_name                                              Decision Tree
config_rank                                                         7
params                          {'criterion': 'gini', 'max_depth': 4}
media_f1_score                                               0.676075
desvio_padrao_f1_score                                       0.029364
media_execution_time                                         1.459164
desvio_padrao_execution_time                                 3.111566
Name: 2, dtype: object
entropy: model                                                              dtree
model_name                                                 Decision Tree
config_rank                                                            4
params                          {'criterion': 'entropy', 'max_depth': 5}
media_f1_score                                                  0.676242
desvio_padrao_f1_score               

### Funçoes para criacao de estimators e models

In [8]:
def create_estimator_from_config(config, params):
    """
    Cria estimador base
    """
    # print("config: ", config)
    # print("params: ", params)
    # print("tipo de params: ", type(params))

    if isinstance(config, list):
        # NB: seleciona classe baseada no nb_type
        nb_type = params.get('nb_type')
        class_map = {
            'GaussianNB': GaussianNB,
            'MultinomialNB': MultinomialNB,
            'ComplementNB': ComplementNB
        }
        model_class = class_map[nb_type]
        return model_class()
    else:
        # Modelos normais: usa parâmetros do grid
        return config(**params)

def create_model_from_config(config, params, estimatorInstance = None):
    """
    Cria modelo baseado na configuração
    """
    print("config: ", config)
    print("params: ", params)
    print("estimatorInstance: ", estimatorInstance)


    actual_params = {k: v for k, v in params.items() if k != 'estimatorModel'}
    
    if (estimatorInstance):
        setup = {
            'estimator': estimatorInstance,
            'n_jobs': paralelismo
        }|actual_params
    else:
        setup = {
            'n_jobs': paralelismo
        }|actual_params
        
    print("config: ", config)
    print("params: ", actual_params)
    print("setup.: ", setup)
    return config['class'](**setup)

par_training = TRAINING_TYPES        

## Define configuração dos classificadores

In [ ]:
MODEL_CONFIGS = {

    'Bagging': {
        'class': BaggingClassifier,
        'param_grid': {
            'n_estimators': [10, 20, 30],
            'estimatorModel': ['knn', 'dtree', 'nb', 'mlp']
        },
        'estimatorClass' : {
            'knn': KNeighborsClassifier,
            'dtree': DecisionTreeClassifier,
            'nb': [GaussianNB, MultinomialNB, ComplementNB],
            'mlp': MLPClassifier
        },
        'fixed_params': {},
        'scale_data': False,
        'warning_exceptions': [],
        'model_name': 'Bagging'
    },
    'Random Forest': {
        'class': RandomForestClassifier,
        'param_grid': {
            'n_estimators': [10, 20, 30, 100],
            'criterion': ['gini', 'entropy', 'log_loss'],
        },
        'fixed_params': {
        },
        'scale_data': False,
        'warning_exceptions': [],
        'model_name': 'Random Forest'
    },
    'Voting': {
        'class': VotingClassifier,
        'param_grid': {
            'n_estimators': [5, 10, 15, 20],
        },
        'estimatorClass' : {
            'knn': KNeighborsClassifier,
            'dtree': DecisionTreeClassifier,
            'mlp': MLPClassifier
        },
        'estimatorConfigPeak': [ 
            {'knn': 1 },
            {'dtree': 1 },
            {'knn': 6 },
            {'mlp': 1 },
            {'mlp': 6 },

            {'knn': 2 },
            {'dtree': 2 },
            {'knn': 8 },  
            {'mlp': 2 },
            {'dtree': 6 },

            {'knn': 3 },
            {'dtree': 3 },
            {'knn': 7 },
            {'mlp': 3 },
            {'dtree': 4 },

            {'knn': 4 },
            {'knn': 5 },
            {'dtree': 5 },
            {'mlp': 4 },
            {'mlp': 5 },
        ],
        'fixed_params': {
            'voting': 'soft'
        },
        'scale_data': False,
        'warning_exceptions': [],
        'model_name': 'Voting'
    },
    'Stacking': {
        'class': StackingClassifier,
        'param_grid': {
            'n_estimators': [5, 10, 15, 20],
        },
        'estimatorClass' : {
            'knn': KNeighborsClassifier,
            'dtree': DecisionTreeClassifier,
            'mlp': MLPClassifier
        },
        'estimatorConfigPeak': [ 
            {'knn': 1 },
            {'dtree': 1 },
            {'knn': 6 },
            {'mlp': 1 },
            {'mlp': 6 },

            {'knn': 2 },
            {'dtree': 2 },
            {'knn': 8 },  
            {'mlp': 2 },
            {'dtree': 6 },

            {'knn': 3 },
            {'dtree': 3 },
            {'knn': 7 },
            {'mlp': 3 },
            {'dtree': 4 },

            {'knn': 4 },
            {'knn': 5 },
            {'dtree': 5 },
            {'mlp': 4 },
            {'mlp': 5 },

        ],
        'fixed_params': {
            'final_estimator': LogisticRegression
        },
        'scale_data': False,
        'warning_exceptions': [],
        'model_name': 'Stacking'
    },

}

## Aplica classificadores em todas as bases de dados

In [10]:
def create_ensemble_estimators(model_key, n_estimators, df_agregado):
    """
    Função auxiliar para criar estimadores para VotingClassifier e StackingClassifier.
    """
    estimator_config_peak = MODEL_CONFIGS[model_key]['estimatorConfigPeak']
    
    # Pegar os primeiros n_estimators elementos da configuração
    configs_to_use = estimator_config_peak[:n_estimators]
    estimators = []
    
    for config_dict in configs_to_use:
        # Cada config_dict é como {'knn': 1}, {'dtree': 2}, etc.
        model_type = list(config_dict.keys())[0]  # 'knn', 'dtree', etc.
        position = config_dict[model_type]        # 1, 2, 3, etc.
        
        # Buscar configuração no dataframe
        model_config = get_nth_best_model(df_agregado, model_type, position, search_by='model')
        
        if model_config is None:
            print(f"Aviso: Configuração não encontrada para {model_type} posição {position}")
            continue
        
        # Criar estimator
        model_name = model_config['model_name']
        config_rank = model_config['config_rank']
        params_dict = ast.literal_eval(model_config['params'])
        
        # Tratar casos especiais (Naive Bayes tem múltiplas classes)
        if model_type == 'nb':
            # Para NB, determinar qual classe usar baseado no nb_type
            nb_type = params_dict.pop('nb_type')  # Remove nb_type dos params
            class_map = {
                'GaussianNB': GaussianNB,
                'MultinomialNB': MultinomialNB,
                'ComplementNB': ComplementNB
            }
            estimator_class = class_map[nb_type]
        else:
            # Para outros modelos, usar estimatorClass diretamente
            estimator_class = MODEL_CONFIGS[model_key]['estimatorClass'][model_type]
        
        estimator = estimator_class(**params_dict)
        estimator_name = f"{model_name} - Config {config_rank}"
        
        estimators.append((estimator_name, estimator))
    
    return estimators

In [11]:
resultados_todos_modelos = []

print("Aplicando classificadores em todas as bases de dados...")
print("=" * 80)

datafile_counter=0
for metadata in datafiles.itertuples():
    datafile_counter += 1
    print(f"\n📊 Dataset {datafile_counter}: {metadata.key}")
    print("-" * 60)

    
    for model_key in MODEL_CONFIGS.keys():
        print(f"\n🔧 Modelo: {MODEL_CONFIGS[model_key]['model_name']} ({model_key}) 📊 Dataset {datafile_counter}: {metadata.key}")
        param_combinations = list(ParameterGrid(MODEL_CONFIGS[model_key]['param_grid']))

        model_config = MODEL_CONFIGS[model_key]

        for config_counter, params in enumerate(param_combinations):
            estimatorSetup = {}
            if model_key == 'Bagging':
                details = get_nth_best_model(df_agregado, params['estimatorModel'], 1, search_by='model')
                # print("details: ", details)
                estimatorSetup = ast.literal_eval(details['params'])

                print(f"Configuração {config_counter+1} 🔧 N.Estimadores: {params['n_estimators']} Estimador Base: {params['estimatorModel']} Setup:{estimatorSetup}")
                estimatorInstance = create_estimator_from_config(MODEL_CONFIGS[model_key]['estimatorClass'][params['estimatorModel']], estimatorSetup)
                print(f"Estimador Instanciado: {estimatorInstance}")

                model = create_model_from_config(MODEL_CONFIGS[model_key], params, estimatorInstance)
            elif model_key == 'Random Forest':
                details = get_best_config_by_criterion(df_agregado, params['criterion'])

                estimatorSetup = ast.literal_eval(details['params'])
                model = create_model_from_config(MODEL_CONFIGS[model_key], estimatorSetup)
            elif model_key == 'Voting':
                # Para VotingClassifier, criar estimadores dinamicamente
                n_estimators = params.get('n_estimators', 5)
                estimators = create_ensemble_estimators(model_key, n_estimators, df_agregado)
                
                # Criar VotingClassifier - usar fixed_params para o voting
                voting_params = {
                    'estimators': estimators,
                    'voting': MODEL_CONFIGS['Voting']['fixed_params']['voting'],
                    'n_jobs': paralelismo
                }
                
                model = VotingClassifier(**voting_params)
                print(f"VotingClassifier criado com {len(estimators)} estimadores: {[name for name, _ in estimators]}")

            elif model_key == 'Stacking':
                # Para StackingClassifier, criar estimadores dinamicamente
                n_estimators = params.get('n_estimators', 5)
                estimators = create_ensemble_estimators(model_key, n_estimators, df_agregado)
                
                # Criar StackingClassifier - usar fixed_params para o final_estimator
                stacking_params = {
                    'estimators': estimators,
                    'final_estimator': MODEL_CONFIGS['Stacking']['fixed_params']['final_estimator'](),
                    'n_jobs': paralelismo
                }
                
                model = StackingClassifier(**stacking_params)
                # print(f"StackingClassifier criado com {len(estimators)} estimadores: {[name for name, _ in estimators]}")
                
            print(f"Modelo Instanciado: {model}")
            for training in par_training:
                print(f"\t\tTraining: {training}")

                print("Parâmetros: ", params)
                scale_data = get_scale_setting(estimatorSetup, metadata.key)
                print("Escalando dados: ", scale_data)

                df = dfs[metadata.key]
                X, y = separar_dataset(df, scale=scale_data)
                
                try:
                    
                    f1 = 0.0
                    f1_std = 0.0
                    confusao = np.array([])
                    execution_time = 0.0
                    
                    start_time = time.time()
                    
                    if training == "holdout":
                        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=tamanho_ds_teste, random_state=semente_aleatoria)
                        
                        with warnings.catch_warnings(record=True) as w:
                            warnings.simplefilter("always")
                            model.fit(X_train, y_train)
                            
                            warning_exceptions = model_config.get('warning_exceptions', [])
                            has_warning = any(issubclass(warning.category, tuple(warning_exceptions)) for warning in w)
                        
                        y_pred = model.predict(X_test)
                        f1 = f1_score(y_test, y_pred, average='weighted')
                        f1_std = 0.0
                        confusao = confusion_matrix(y_test, y_pred)
                        
                    elif training == "crossvalidation":
                        kf = KFold(n_splits=10, random_state=semente_aleatoria, shuffle=True)
                        
                        with warnings.catch_warnings(record=True) as w:
                            warnings.simplefilter("always")
                            scores = cross_val_score(model, X, y, scoring='f1_weighted', cv=kf, n_jobs = paralelismo)
                            y_pred = cross_val_predict(model, X, y, cv=kf, n_jobs = paralelismo)
                            
                            warning_exceptions = model_config.get('warning_exceptions', [])
                            has_warning = any(issubclass(warning.category, tuple(warning_exceptions)) for warning in w)
                        
                        confusao = confusion_matrix(y, y_pred)
                        f1 = scores.mean()
                        f1_std = scores.std()
                    
                    execution_time = time.time() - start_time
                    
                    if training == "holdout":
                        if has_warning:
                            print(f"\t\t\t⚠ WARNING: Problema detectado - F1: {f1:.4f} (Time: {execution_time:.4f}s)")
                        else:
                            print(f"\t\t\t✓ Sucesso - F1: {f1:.4f} (Time: {execution_time:.4f}s)")
                    else:
                        if has_warning:
                            print(f"\t\t\t⚠ WARNING: Problema detectado - F1: {f1:.4f} ({f1_std:.4f}) (Time: {execution_time:.4f}s)")
                        else:
                            print(f"\t\t\t✓ Sucesso - F1: {f1:.4f} ({f1_std:.4f}) (Time: {execution_time:.4f}s)")
                    
                    resultado = {
                        'dataset': metadata.key,
                        'model': model_key,
                        'model_name': model_config['model_name'],
                        'config_rank': config_counter+1,
                        'params': params,
                        'training_type': training,
                        'f1_score': f1,
                        'f1_std': f1_std,
                        'confusion_matrix': confusao,
                        'trained_model': model,
                        'scale_data': scale_data,
                        'execution_time': execution_time
                    }
                    resultados_todos_modelos.append(resultado)
                    
                except Exception as e:
                    print(f"\t\t\t✗ ERRO: {type(e).__name__}: {str(e)}")
                    continue
    


Aplicando classificadores em todas as bases de dados...

📊 Dataset 1: hogfeat_128_16_4_9_pca
------------------------------------------------------------

🔧 Modelo: Bagging (Bagging) 📊 Dataset 1: hogfeat_128_16_4_9_pca
Configuração 1 🔧 N.Estimadores: 10 Estimador Base: knn Setup:{'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'uniform'}
Estimador Instanciado: KNeighborsClassifier(metric='chebyshev', n_neighbors=3)
config:  {'class': <class 'sklearn.ensemble._bagging.BaggingClassifier'>, 'param_grid': {'n_estimators': [10, 20, 30], 'estimatorModel': ['knn', 'dtree', 'nb', 'mlp']}, 'estimatorClass': {'knn': <class 'sklearn.neighbors._classification.KNeighborsClassifier'>, 'dtree': <class 'sklearn.tree._classes.DecisionTreeClassifier'>, 'nb': [<class 'sklearn.naive_bayes.GaussianNB'>, <class 'sklearn.naive_bayes.MultinomialNB'>, <class 'sklearn.naive_bayes.ComplementNB'>], 'mlp': <class 'sklearn.neural_network._multilayer_perceptron.MLPClassifier'>}, 'fixed_params': {}, 'scale_data':

In [14]:
resultados_todos_modelos

[{'dataset': 'hogfeat_128_16_4_9_pca',
  'model': 'Bagging',
  'model_name': 'Bagging',
  'config_rank': 1,
  'params': {'estimatorModel': 'knn', 'n_estimators': 10},
  'training_type': 'holdout',
  'f1_score': 0.7485613810741689,
  'f1_std': 0.0,
  'confusion_matrix': array([[48, 23],
         [17, 72]]),
  'trained_model': BaggingClassifier(estimator=KNeighborsClassifier(metric='chebyshev',
                                                   n_neighbors=3),
                    n_jobs=-1),
  'scale_data': False,
  'execution_time': 0.4096720218658447},
 {'dataset': 'hogfeat_128_16_4_9_pca',
  'model': 'Bagging',
  'model_name': 'Bagging',
  'config_rank': 1,
  'params': {'estimatorModel': 'knn', 'n_estimators': 10},
  'training_type': 'crossvalidation',
  'f1_score': np.float64(0.7186624123580465),
  'f1_std': np.float64(0.0301406966789107),
  'confusion_matrix': array([[262, 138],
         [ 85, 315]]),
  'trained_model': BaggingClassifier(estimator=KNeighborsClassifier(metric='chebys

In [18]:
resultados_todos_modelos
print("💾 Resultados salvos:")

# Converter a lista para DataFrame primeiro
df_csv_resultados = pd.DataFrame(resultados_todos_modelos)

# Remover colunas não serializáveis
df_csv_resultados = df_csv_resultados.drop('confusion_matrix', axis=1)
df_csv_resultados = df_csv_resultados.drop('trained_model', axis=1)

# Salvar como CSV
output_path = 'resultados_todos_modelos.csv'
df_csv_resultados.to_csv(output_path, index=False)
print(f"Arquivo CSV salvo com sucesso em: {output_path}")
print(f"Shape: {df_csv_resultados.shape}")
print(f"Colunas: {list(df_csv_resultados.columns)}")


# Salvar resultados completos com joblib
joblib.dump(resultados_todos_modelos, f'resultados_top10_todos_comites_{versao_base}.joblib')
print(f"- resultados_top10_todos_comites_.joblib: dados completos com modelos treinados")

💾 Resultados salvos:
Arquivo CSV salvo com sucesso em: resultados_todos_modelos.csv
Shape: (768, 10)
Colunas: ['dataset', 'model', 'model_name', 'config_rank', 'params', 'training_type', 'f1_score', 'f1_std', 'scale_data', 'execution_time']
- resultados_top10_todos_comites_.joblib: dados completos com modelos treinados


In [ ]:
print("📈 Melhores F1 scores por dataset e modelo:")
for dataset in df_resultados_todos['dataset'].unique():
    print(f"\n  Dataset: {dataset}")
    dataset_results = df_resultados_todos[df_resultados_todos['dataset'] == dataset]
    for model in dataset_results['model'].unique():
        model_results = dataset_results[dataset_results['model'] == model]
        best_result = model_results.loc[model_results['f1_score'].idxmax()]
        print(f"    {model.upper()}: {best_result['f1_score']:.4f} (Config {best_result['config_rank']}, {best_result['training_type']})")
